In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout, LeakyReLU, Conv2D, Conv2DTranspose, BatchNormalization, Concatenate, Add, Activation, GlobalAveragePooling2D, multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import cv2

np.random.seed(42)
tf.random.set_seed(42)

normal_path = r"C:\Users\T2410216\Desktop\Model1\Datasets\Augmented\Brain_AD\Normal"
abnormal_path = r"C:\Users\T2410216\Desktop\Model1\Datasets\Augmented\Brain_AD\Abnormal"


IMG_SIZE = (128, 128)


def load_images(path, img_size=IMG_SIZE, normalization="[0, 1]", max_images=5000):
    images = []
    image_files = glob.glob(os.path.join(path, "*.png")) + glob.glob(os.path.join(path, "*.jpg"))


    if not os.path.exists(path) or not image_files:
        print(f"Path {path} not found or no images found. Creating dummy data.")
        if "normal" in path.lower():
            return create_dummy_normal_data(100, img_size)
        else:
            return create_dummy_abnormal_data(50, img_size)

    print(f"Found {len(image_files)} images in {path}")


    image_files = image_files[:max_images]


    for img_path in image_files:
        try:
            img = Image.open(img_path).convert('L')
            img = img.resize(img_size)


            img_array = np.array(img).astype('float32')


            if normalization == "[0, 1]":
                img_array = img_array / 255.0
            elif normalization == "[-1, 1]":
                img_array = (img_array / 127.5) - 1.0

            images.append(img_array)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

    return np.array(images)


def create_dummy_normal_data(n_samples=100, img_size=IMG_SIZE):
    normal_images = np.random.rand(n_samples, img_size[0], img_size[1]) * 0.2

    for i in range(n_samples):
        center = (img_size[0]//2, img_size[1]//2)
        for x in range(img_size[0]):
            for y in range(img_size[1]):
                dist = np.sqrt((x - center[0])**2 + (y - center[1])**2)
                if dist < 40:
                    normal_images[i, x, y] += 0.5
    return normal_images

def create_dummy_abnormal_data(n_samples=50, img_size=IMG_SIZE):

    abnormal_images = create_dummy_normal_data(n_samples, img_size)

    for i in range(n_samples):
        x_anom = np.random.randint(img_size[0]//4, 3*img_size[0]//4)
        y_anom = np.random.randint(img_size[1]//4, 3*img_size[1]//4)
        abnormal_images[i, x_anom-10:x_anom+10, y_anom-10:y_anom+10] = 1.0
    return abnormal_images


print("Loading normal images...")
normal_images = load_images(normal_path, img_size=IMG_SIZE)
print("Loading abnormal images...")
abnormal_images = load_images(abnormal_path, img_size=IMG_SIZE)

print(f"Loaded {len(normal_images)} normal images and {len(abnormal_images)} abnormal images")


normal_labels = np.zeros(len(normal_images))
abnormal_labels = np.ones(len(abnormal_images))


x_train_normal, x_test_normal = train_test_split(normal_images, test_size=0.2, random_state=42)

x_test = np.concatenate([x_test_normal, abnormal_images])
y_test = np.concatenate([np.zeros(len(x_test_normal)), np.ones(len(abnormal_images))])


LATENT_DIM = 128
IMG_SHAPE = (128, 128, 1)

def attention_block(x, filters):
    se = GlobalAveragePooling2D()(x)
    se = Dense(filters // 16, activation='relu')(se)
    se = Dense(filters, activation='sigmoid')(se)
    se = Reshape((1, 1, filters))(se)

    x = multiply([x, se])
    return x


def build_encoder():
    img_input = Input(shape=IMG_SHAPE)

    x = Conv2D(32, kernel_size=4, strides=2, padding='same')(img_input)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2D(64, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 64)

    x = Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 128)

    x = Conv2D(256, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 256)

    x = Flatten()(x)
    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    z = Dense(LATENT_DIM)(x)

    return Model(img_input, z, name='encoder')


def build_generator():
    z_input = Input(shape=(LATENT_DIM,))


    x = Dense(8 * 8 * 256)(z_input)
    x = LeakyReLU(alpha=0.2)(x)
    x = Reshape((8, 8, 256))(x)


    x = Conv2DTranspose(256, kernel_size=4, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 256)


    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 128)


    skip1 = x
    x = Conv2D(128, kernel_size=3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Conv2D(128, kernel_size=3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Add()([x, skip1])


    x = Conv2DTranspose(64, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = attention_block(x, 64)


    skip2 = x
    x = Conv2D(64, kernel_size=3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Conv2D(64, kernel_size=3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Add()([x, skip2])


    x = Conv2DTranspose(32, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)


    x = Conv2DTranspose(16, kernel_size=4, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2D(1, kernel_size=3, strides=1, padding='same', activation='sigmoid')(x)

    return Model(z_input, x, name='generator')


def build_discriminator():
    img_input = Input(shape=IMG_SHAPE)
    z_input = Input(shape=(LATENT_DIM,))


    x1 = Conv2D(32, kernel_size=4, strides=2, padding='same')(img_input)
    x1 = LeakyReLU(alpha=0.2)(x1)

    x1 = Conv2D(64, kernel_size=4, strides=2, padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = LeakyReLU(alpha=0.2)(x1)

    x1 = Conv2D(128, kernel_size=4, strides=2, padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = LeakyReLU(alpha=0.2)(x1)

    x1 = Conv2D(256, kernel_size=4, strides=2, padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = LeakyReLU(alpha=0.2)(x1)

    x1 = Flatten()(x1)


    x2 = Dense(512)(z_input)
    x2 = LeakyReLU(alpha=0.2)(x2)


    x = Concatenate()([x1, x2])

    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.5)(x)

    x = Dense(1, activation='sigmoid')(x)

    return Model([img_input, z_input], x, name='discriminator')


def build_bigan(generator, encoder, discriminator):
    discriminator.trainable = False
    z = Input(shape=(LATENT_DIM,))
    fake_img = generator(z)
    img = Input(shape=IMG_SHAPE)
    encoded_z = encoder(img)
    fake_pair_validity = discriminator([fake_img, z])
    real_pair_validity = discriminator([img, encoded_z])
    bigan = Model([z, img], [fake_pair_validity, real_pair_validity])


    bigan.compile(loss=['binary_crossentropy', 'binary_crossentropy'],
                  optimizer=Adam(learning_rate=0.00005, beta_1=0.5),
                  loss_weights=[1, 1])

    return bigan


def build_perceptual_model():
    vgg = VGG16(include_top=False, weights='imagenet', input_shape=(128, 128, 3))


    for layer in vgg.layers:
        layer.trainable = False

    feature_outputs = [vgg.get_layer('block1_conv2').output,
                      vgg.get_layer('block2_conv2').output,
                      vgg.get_layer('block3_conv3').output]

    return Model(vgg.input, feature_outputs)


def ssim_loss(y_true, y_pred):
    return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, 1.0))


def build_reconstruction_model(encoder, generator, perceptual_model=None):
    img_input = Input(shape=IMG_SHAPE)
    z = encoder(img_input)
    reconstructed_img = generator(z)
    reconstruction_model = Model(img_input, reconstructed_img)

    if perceptual_model:
        def combined_loss(y_true, y_pred):
            y_true_rgb = tf.concat([y_true, y_true, y_true], axis=-1)
            y_pred_rgb = tf.concat([y_pred, y_pred, y_pred], axis=-1)

            y_true_rgb = preprocess_input(y_true_rgb * 255.0)
            y_pred_rgb = preprocess_input(y_pred_rgb * 255.0)

            true_features = perceptual_model(y_true_rgb)
            pred_features = perceptual_model(y_pred_rgb)

            perceptual_loss = 0
            for true_feat, pred_feat in zip(true_features, pred_features):
                perceptual_loss += tf.reduce_mean(tf.square(true_feat - pred_feat))

            mse = tf.reduce_mean(tf.square(y_true - y_pred))
            mae = tf.reduce_mean(tf.abs(y_true - y_pred))
            ssim = ssim_loss(y_true, y_pred)

            return 0.5 * mse + 0.2 * mae + 0.2 * perceptual_loss + 0.1 * ssim

        reconstruction_model.compile(loss=combined_loss,
                                    optimizer=Adam(learning_rate=0.0001, beta_1=0.5))
    else:
        def combined_loss(y_true, y_pred):
            mse = tf.reduce_mean(tf.square(y_true - y_pred))
            mae = tf.reduce_mean(tf.abs(y_true - y_pred))
            ssim = ssim_loss(y_true, y_pred)
            return 0.6 * mse + 0.3 * mae + 0.1 * ssim

        reconstruction_model.compile(loss=combined_loss,
                                    optimizer=Adam(learning_rate=0.0001, beta_1=0.5))

    return reconstruction_model


def train_bigan_with_reconstruction(bigan, generator, encoder, discriminator, reconstruction_model, x_train, epochs=500, batch_size=32):
    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    os.makedirs('progress_images', exist_ok=True)

    # Phase 1
    print("Phase 1: Pre-training reconstruction...")
    for epoch in range(150):
        idx = np.random.randint(0, x_train.shape[0], batch_size)
        imgs = x_train[idx]

        r_loss = reconstruction_model.train_on_batch(imgs, imgs)

        if epoch % 10 == 0:
            print(f"Pre-training Epoch {epoch}/150, [R loss: {r_loss:.4f}]")

            if epoch % 20 == 0:
                sample_imgs = x_train[np.random.randint(0, x_train.shape[0], 5)]
                encoded_z = encoder.predict(sample_imgs)
                reconstructed_imgs = generator.predict(encoded_z)

                plt.figure(figsize=(10, 4))
                for i in range(5):
                    plt.subplot(2, 5, i+1)
                    plt.imshow(sample_imgs[i].reshape(IMG_SHAPE[:2]), cmap='gray')
                    plt.axis('off')
                    plt.title("Original")

                    plt.subplot(2, 5, i+6)
                    plt.imshow(reconstructed_imgs[i].reshape(IMG_SHAPE[:2]), cmap='gray')
                    plt.axis('off')
                    plt.title("Reconstruction")

                plt.tight_layout()
                plt.savefig(f'progress_images/pretraining_epoch_{epoch}.png')
                plt.close()

    # Phase 2
    print("Phase 2: Joint training with emphasis on reconstruction...")
    for epoch in range(epochs):
        if epoch % 5 == 0:
            idx = np.random.randint(0, x_train.shape[0], batch_size)
            imgs = x_train[idx]

            z = np.random.normal(0, 1, (batch_size, LATENT_DIM))
            fake_imgs = generator.predict(z)

            encoded_z = encoder.predict(imgs)

            d_loss_real = discriminator.train_on_batch([imgs, encoded_z], valid)
            d_loss_fake = discriminator.train_on_batch([fake_imgs, z], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        else:
            d_loss = [0, 0]


        idx = np.random.randint(0, x_train.shape[0], batch_size)
        imgs = x_train[idx]

        z = np.random.normal(0, 1, (batch_size, LATENT_DIM))

        g_loss = bigan.train_on_batch([z, imgs], [valid, fake])

        for _ in range(3):
            idx = np.random.randint(0, x_train.shape[0], batch_size)
            imgs = x_train[idx]
            r_loss = reconstruction_model.train_on_batch(imgs, imgs)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs}, [D loss: {d_loss[0]:.4f}] [G/E loss: {g_loss[0]:.4f}] [R loss: {r_loss:.4f}]")

            if epoch % 50 == 0:
                sample_imgs = x_train[np.random.randint(0, x_train.shape[0], 5)]
                encoded_z = encoder.predict(sample_imgs)
                reconstructed_imgs = generator.predict(encoded_z)

                plt.figure(figsize=(10, 4))
                for i in range(5):
                    plt.subplot(2, 5, i+1)
                    plt.imshow(sample_imgs[i].reshape(IMG_SHAPE[:2]), cmap='gray')
                    plt.axis('off')
                    plt.title("Original")

                    plt.subplot(2, 5, i+6)
                    plt.imshow(reconstructed_imgs[i].reshape(IMG_SHAPE[:2]), cmap='gray')
                    plt.axis('off')
                    plt.title("Reconstruction")

                plt.tight_layout()
                plt.savefig(f'progress_images/joint_training_epoch_{epoch}.png')
                plt.close()


def calculate_anomaly_scores(x_test, encoder, generator):
    encoded_imgs = encoder.predict(x_test)
    reconstructed_imgs = generator.predict(encoded_imgs)

    mse_errors = np.mean(np.square(x_test - reconstructed_imgs), axis=(1, 2, 3))

    ssim_scores = []
    for i in range(len(x_test)):
        ssim = tf.image.ssim(
            tf.convert_to_tensor(x_test[i:i+1]),
            tf.convert_to_tensor(reconstructed_imgs[i:i+1]),
            max_val=1.0
        ).numpy()[0]
        ssim_scores.append(1.0 - ssim)

    ssim_scores = np.array(ssim_scores)

    combined_scores = 0.7 * mse_errors + 0.3 * ssim_scores

    return combined_scores, reconstructed_imgs

def evaluate_anomaly_detection(anomaly_scores, y_test):
    fpr, tpr, thresholds = roc_curve(y_test, anomaly_scores)
    roc_auc = auc(fpr, tpr)

    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, anomaly_scores)
    average_precision = average_precision_score(y_test, anomaly_scores)

    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]

    predictions = (anomaly_scores >= optimal_threshold).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    specificity = tn / (tn + fp)

    print("\n===== ANOMALY DETECTION PERFORMANCE =====")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision: {average_precision:.4f}")
    print(f"Optimal Threshold: {optimal_threshold:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print("=========================================\n")

    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(2, 2, 2)
    plt.plot(recall_curve, precision_curve, color='blue', lw=2, label=f'AP = {average_precision:.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="upper right")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0.0, 1.05])
    plt.xlim([0.0, 1.0])


    plt.subplot(2, 2, 3)
    plt.hist(anomaly_scores[y_test == 0], bins=30, alpha=0.5, label='Normal', color='green')
    plt.hist(anomaly_scores[y_test == 1], bins=30, alpha=0.5, label='Abnormal', color='red')
    plt.axvline(x=optimal_threshold, color='black', linestyle='--',
                label=f'Threshold: {optimal_threshold:.3f}')
    plt.xlabel('Anomaly Score')
    plt.ylabel('Count')
    plt.title('Distribution of Anomaly Scores')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(2, 2, 4)
    cm = confusion_matrix(y_test, predictions)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ['Normal', 'Abnormal'], rotation=45)
    plt.yticks(tick_marks, ['Normal', 'Abnormal'])

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    plt.tight_layout()
    plt.savefig('evaluation_metrics.png')
    plt.show()

    plt.figure(figsize=(10, 6))
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Specificity', 'ROC AUC']
    values = [accuracy, precision, recall, f1, specificity, roc_auc]

    plt.bar(metrics, values, color=['blue', 'green', 'red', 'purple', 'orange', 'teal'])
    plt.ylim([0, 1.1])
    plt.ylabel('Score')
    plt.title('Anomaly Detection Performance Metrics')

    for i, v in enumerate(values):
        plt.text(i, v + 0.02, f'{v:.3f}', ha='center')

    plt.tight_layout()
    plt.savefig('metrics_summary.png')
    plt.show()

    return {
        'roc_auc': roc_auc,
        'average_precision': average_precision,
        'optimal_threshold': optimal_threshold,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1_score': f1,
        'confusion_matrix': {
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp)
        }
    }

def visualize_reconstructions(x_test, y_test, reconstructed_imgs, anomaly_scores, threshold, num_examples=5):
    normal_indices = np.where(y_test == 0)[0]
    abnormal_indices = np.where(y_test == 1)[0]

    np.random.shuffle(normal_indices)
    np.random.shuffle(abnormal_indices)

    normal_examples = normal_indices[:num_examples]
    abnormal_examples = abnormal_indices[:num_examples]

    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(normal_examples):
        plt.subplot(2, num_examples, i + 1)
        plt.imshow(x_test[idx].reshape(IMG_SHAPE[:2]), cmap='gray')
        plt.title(f"Normal\nScore: {anomaly_scores[idx]:.4f}")
        plt.axis('off')

        plt.subplot(2, num_examples, i + 1 + num_examples)
        plt.imshow(reconstructed_imgs[idx].reshape(IMG_SHAPE[:2]), cmap='gray')
        plt.title("Reconstruction")
        plt.axis('off')

    plt.tight_layout()
    plt.savefig('normal_reconstructions.png')
    plt.show()


    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(abnormal_examples):
        plt.subplot(2, num_examples, i + 1)
        plt.imshow(x_test[idx].reshape(IMG_SHAPE[:2]), cmap='gray')
        plt.title(f"Abnormal\nScore: {anomaly_scores[idx]:.4f}")
        plt.axis('off')

        plt.subplot(2, num_examples, i + 1 + num_examples)
        plt.imshow(reconstructed_imgs[idx].reshape(IMG_SHAPE[:2]), cmap='gray')
        plt.title("Reconstruction")
        plt.axis('off')

    plt.tight_layout()
    plt.savefig('abnormal_reconstructions.png')
    plt.show()

def save_models(encoder, generator, discriminator, save_dir="bigan_models"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    encoder.save(os.path.join(save_dir, "encoder.h5"))
    generator.save(os.path.join(save_dir, "generator.h5"))
    discriminator.save(os.path.join(save_dir, "discriminator.h5"))
    print(f"Models saved to {save_dir}")


def main():
    x_train_normal_reshaped = x_train_normal.reshape(-1, IMG_SHAPE[0], IMG_SHAPE[1], 1)
    x_test_reshaped = x_test.reshape(-1, IMG_SHAPE[0], IMG_SHAPE[1], 1)

    encoder = build_encoder()
    generator = build_generator()
    discriminator = build_discriminator()

    discriminator.compile(loss='binary_crossentropy',
                         optimizer=Adam(learning_rate=0.00005, beta_1=0.5),
                         metrics=['accuracy'])

    bigan = build_bigan(generator, encoder, discriminator)

    try:
        perceptual_model = build_perceptual_model()
        print("Using perceptual loss with VGG16")
    except:
        perceptual_model = None
        print("VGG16 not available, using standard loss")

    reconstruction_model = build_reconstruction_model(encoder, generator, perceptual_model)

    print("Encoder Summary:")
    encoder.summary()
    print("\nGenerator Summary:")
    generator.summary()
    print("\nDiscriminator Summary:")
    discriminator.summary()

    train_bigan_with_reconstruction(bigan, generator, encoder, discriminator, reconstruction_model,
                                   x_train_normal_reshaped, epochs=500, batch_size=32)

    anomaly_scores, reconstructed_imgs = calculate_anomaly_scores(x_test_reshaped, encoder, generator)
    metrics = evaluate_anomaly_detection(anomaly_scores, y_test)

    visualize_reconstructions(x_test_reshaped, y_test, reconstructed_imgs, anomaly_scores, metrics['optimal_threshold'])

    save_models(encoder, generator, discriminator)

    with open('anomaly_detection_metrics.txt', 'w') as f:
        f.write("===== ANOMALY DETECTION PERFORMANCE =====\n")
        f.write(f"ROC AUC: {metrics['roc_auc']:.4f}\n")
        f.write(f"Average Precision: {metrics['average_precision']:.4f}\n")
        f.write(f"Optimal Threshold: {metrics['optimal_threshold']:.4f}\n")
        f.write(f"Accuracy: {metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {metrics['precision']:.4f}\n")
        f.write(f"Recall (Sensitivity): {metrics['recall']:.4f}\n")
        f.write(f"Specificity: {metrics['specificity']:.4f}\n")
        f.write(f"F1 Score: {metrics['f1_score']:.4f}\n")
        f.write(f"Confusion Matrix: TN={metrics['confusion_matrix']['tn']}, ")
        f.write(f"FP={metrics['confusion_matrix']['fp']}, ")
        f.write(f"FN={metrics['confusion_matrix']['fn']}, ")
        f.write(f"TP={metrics['confusion_matrix']['tp']}\n")
        f.write("=========================================\n")

    return encoder, generator, discriminator, anomaly_scores, metrics

if __name__ == "__main__":
    encoder, generator, discriminator, anomaly_scores, metrics = main()

Loading normal images...
Path C:\Users\T2410216\Desktop\Model1\Datasets\Augmented\Brain_AD\Normal not found or no images found. Creating dummy data.
Loading abnormal images...
Path C:\Users\T2410216\Desktop\Model1\Datasets\Augmented\Brain_AD\Abnormal not found or no images found. Creating dummy data.
Loaded 100 normal images and 100 abnormal images


c:\Users\Ashim\anaconda3\envs\paper\lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 25s 0us/step
Using perceptual loss with VGG16
Encoder Summary:


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        544 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     32,832 │ leaky_re_lu[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ leaky_re_lu_1[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │        260 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        320 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 64)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 32,    │          0 │ leaky_re_lu_1[0]… │
│                     │ 64)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │    131,200 │ multiply[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ leaky_re_lu_2[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │      1,032 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │      1,152 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1, 128) │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 16, 16,    │          0 │ leaky_re_lu_2[0]

...

 Total params: 9,408,737 (35.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 9,408,737 (35.89 MB)

Phase 1: Pre-training reconstruction...
Pre-training Epoch 0/150, [R loss: 86677.9297]
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


KeyboardInterrupt: 